# Regression with tidymodels

This lesson is adapted from Chapter 10, "Regression", of:

> Downey, A. *Elements of Data Science.* <https://allendowney.github.io/ElementsOfDataScience/10_regression.html>

Licensed CC BY-NC-SA 4.0. The original uses Python and StatsModels. We do the same analysis in R with [tidymodels](https://www.tidymodels.org/).

## Setup

**tidymodels** is a set of R packages for fitting models the tidyverse way. You installed it with `pixi add r-tidymodels`. If the cell below fails, go back to the [guide](https://unc-bios-512.github.io/regression.html).

In [ ]:
library('tidymodels')

## The data

We are using the [GSS](https://gss.norc.org/) data again. The General Social Survey has asked adults in the United States the same questions since 1972. Our subset has these columns:

| Column | Meaning |
| --- | --- |
| `year` | survey year |
| `age` | age in years |
| `educ` | years of school |
| `sex` | 1 = male, 2 = female |
| `realinc` | household income, in 1986 dollars |

In [ ]:
gss <- read.csv('https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/gss_subset.csv')

gss |> glimpse()

Models cannot use missing values. Drop rows with `NA` in the columns we need. Turn `sex` into a **factor** with labels, so R knows it is a category and not a number.

In [ ]:
gss <- gss |>
    drop_na(realinc, educ, age, sex) |>
    mutate(sex = factor(sex, levels = c(1, 2), labels = c('male', 'female')))

gss |> count(sex)

## 10.1 Simple regression

How does income change with years of school? Look first. There are many people at each education level, so plot the mean income for each.

In [ ]:
gss |>
    group_by(educ) |>
    summarize(realinc = mean(realinc)) |>
ggplot(aes(x = educ, y = realinc)) + geom_point()

A **linear regression** finds the straight line that fits best.

In tidymodels you say what kind of model you want, `linear_reg()`, then `fit()` it with a **formula**. The formula `realinc ~ educ` means: predict `realinc` from `educ`.

In [ ]:
fit1 <- linear_reg() |> fit(realinc ~ educ, data = gss)

fit1

In [ ]:
fit1 |> tidy()

`tidy()` gives the coefficients as a table.

- The `educ` estimate is the **slope**. Each extra year of school goes with about $3,600 more income.
- The `(Intercept)` is the predicted income at 0 years of school. It is negative, which makes no sense. That is OK. Almost nobody in the data is near 0, so the line is not meant to be used there.

**You do:** fit the same kind of model with `age` instead of `educ`. What is the slope?

<details>
<summary>Answer</summary>

```r
fit_age <- linear_reg() |> fit(realinc ~ age, data = gss)

fit_age |> tidy()
```

</details>

## 10.2 Multiple regression

We can use more than one predictor. Add them with `+`.

In [ ]:
fit2 <- linear_reg() |> fit(realinc ~ educ + age, data = gss)

fit2 |> tidy()

Now each slope means "holding the other one fixed".

- Holding age fixed, a year of school is worth about $3,700.
- Holding school fixed, a year of age is worth about $40.

## 10.3 Grouping by age

Is income really a straight line in age? Plot the mean income at each age.

In [ ]:
gss |>
    group_by(age) |>
    summarize(realinc = mean(realinc)) |>
ggplot(aes(x = age, y = realinc)) + geom_point()

Not a line. Income goes up until about 50, then down.

A straight line cannot bend. But a line plus a **squared** term can. Add a column `age2 = age^2` and put it in the model.

In [ ]:
gss <- gss |> mutate(age2 = age^2)

fit3 <- linear_reg() |> fit(realinc ~ educ + age + age2, data = gss)

fit3 |> tidy()

The `age` slope is positive and the `age2` slope is negative. That makes a hill shape: up, then down.

**You do:** does the same trick help with education? Add `educ2 = educ^2` to `gss` and fit `realinc ~ educ + educ2 + age + age2`. Look back at the mean-income-by-education plot. Does a curve make sense there?

<details>
<summary>Answer</summary>

```r
gss <- gss |> mutate(educ2 = educ^2)

fit4 <- linear_reg() |> fit(realinc ~ educ + educ2 + age + age2, data = gss)

fit4 |> tidy()
```

</details>

## 10.4 Visualizing predictions

Coefficients are hard to read. Pictures are easier.

First make sure we have the model with both squared terms.

In [ ]:
gss <- gss |> mutate(educ2 = educ^2)

fit4 <- linear_reg() |> fit(realinc ~ educ + educ2 + age + age2, data = gss)

fit4 |> tidy()

Now ask the model to **predict** income for made-up people: every age from 18 to 89, with 12, 14, or 16 years of school. `expand_grid()` makes every combination.

In [ ]:
grid <- expand_grid(age = 18:89, educ = c(12, 14, 16)) |>
    mutate(age2 = age^2, educ2 = educ^2)

grid |> head()

`predict()` takes the model and the new data. It returns one column, `.pred`. Glue it onto the grid with `bind_cols()`.

In [ ]:
preds <- fit4 |> predict(new_data = grid) |> bind_cols(grid)

preds |> head()

In [ ]:
income_by_age <- gss |>
    group_by(age) |>
    summarize(realinc = mean(realinc))

ggplot() +
    geom_point(data = income_by_age, aes(x = age, y = realinc), alpha = 0.5) +
    geom_line(data = preds, aes(x = age, y = .pred, color = factor(educ))) +
    labs(color = 'educ')

**You do:** make a grid with `educ = 0:20` and `age = 30`. Do not forget `age2` and `educ2`. Predict with `fit4`. Then draw the line over the mean-income-by-education points.

<details>
<summary>Answer</summary>

```r
grid_educ <- expand_grid(educ = 0:20, age = 30) |>
    mutate(age2 = age^2, educ2 = educ^2)

preds_educ <- fit4 |> predict(new_data = grid_educ) |> bind_cols(grid_educ)

income_by_educ <- gss |>
    group_by(educ) |>
    summarize(realinc = mean(realinc))

ggplot() +
    geom_point(data = income_by_educ, aes(x = educ, y = realinc), alpha = 0.5) +
    geom_line(data = preds_educ, aes(x = educ, y = .pred))
```

</details>

## 10.5 Categorical variables

`sex` is not a number. It is a **category**. R can still use it in a formula.

In [ ]:
fit5 <- linear_reg() |> fit(realinc ~ educ + educ2 + age + age2 + sex, data = gss)

fit5 |> tidy()

The row is called `sexfemale`, not `sex`. Its estimate is about −$4,900. That means: holding age and school fixed, women report about $4,900 less income than men.

Careful: `realinc` is **household** income, not personal income. So this is not a direct measure of the gender pay gap.

### How R encodes a factor

A model can only multiply numbers. So R turns `sex` into a 0/1 column behind the scenes. This is called a **dummy variable**.

The first level, `male`, is the **baseline**. It gets no column. The column `sexfemale` is 1 for women and 0 for men. Let's look.

In [ ]:
contrasts(gss$sex)

That is the lookup table. Now see the columns R builds for the first few rows. Compare them with the `sex` column.

In [ ]:
head(gss) |> select(sex)

In [ ]:
model.matrix(~ sex, data = head(gss))

And here is the full table the model actually used. tidymodels wraps a plain `lm` object. `extract_fit_engine()` pulls it out.

In [ ]:
fit5 |> extract_fit_engine() |> model.matrix() |> head()

Each coefficient in `tidy(fit5)` multiplies one of these columns. The `sexfemale` coefficient is just "how much to add when that column is 1".

tidymodels can also make dummy columns by hand with `recipe() |> step_dummy()`. Here the formula does it for us.

**You do:** what if `sex` were a plain number (1 and 2) instead of a factor? Refit with `as.numeric(sex)` and compare the `tidy()` tables.

<details>
<summary>Answer</summary>

```r
fit5b <- linear_reg() |> fit(realinc ~ educ + educ2 + age + age2 + as.numeric(sex), data = gss)

fit5b |> tidy()

# The slope is the same size. Only the intercept moves, because the
# baseline is now 0 instead of 1. With two levels that works. With three
# or more levels a plain number would force them onto a line in a made-up
# order. A factor avoids that.
```

</details>

## Turn in

You turn in this notebook, with the outputs from live coding in class.

1. Save the notebook with **Ctrl+S**. Every cell should show its output.
2. Commit and push to your fork. See steps 6 and 7 of the [guide](https://unc-bios-512.github.io/regression.html).
3. Open the notebook on GitHub and paste its link into the Canvas assignment.